In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Paths
CK_TRAIN = '/content/drive/MyDrive/FER_project/data/CK/train'
CK_TEST  = '/content/drive/MyDrive/FER_project/data/CK/test'
FER_TRAIN = '/content/drive/MyDrive/FER_project/data/FER-2013/train'
FER_TEST  = '/content/drive/MyDrive/FER_project/data/FER-2013/test'
RESULTS   = '/content/drive/MyDrive/FER_project/RESULTS'

os.makedirs(RESULTS, exist_ok=True)
print("Paths set. RESULTS folder ready at:", RESULTS)

Mounted at /content/drive
Paths set. RESULTS folder ready at: /content/drive/MyDrive/FER_project/RESULTS


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import json

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten,
                                     Dense, Dropout, GlobalAveragePooling2D)
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay)

IMG_SIZE   = 48
BATCH_SIZE = 32
EPOCHS     = 30
NUM_CLASSES = 6

print("All imports done.")

All imports done.


In [ ]:
# ── CK+ Data Generators ──────────────────────────────────────────────────────
ck_train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.15,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

ck_test_gen = ImageDataGenerator(rescale=1./255)

ck_train = ck_train_gen.flow_from_directory(
    CK_TRAIN,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

ck_test = ck_test_gen.flow_from_directory(
    CK_TEST,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("CK+ class indices:", ck_train.class_indices)
print(f"CK+ — Train samples: {ck_train.samples}, Test samples: {ck_test.samples}")

Found 341 images belonging to 6 classes.
Found 143 images belonging to 6 classes.
CK+ class indices: {'anger': 0, 'fear': 1, 'happy': 2, 'neutral': 3, 'sadness': 4, 'surprise': 5}
CK+ — Train samples: 341, Test samples: 143


In [ ]:
# ── FER-2013 Data Generators ─────────────────────────────────────────────────
fer_train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.15,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

fer_test_gen = ImageDataGenerator(rescale=1./255)

fer_train = fer_train_gen.flow_from_directory(
    FER_TRAIN,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

fer_test = fer_test_gen.flow_from_directory(
    FER_TEST,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("FER-2013 class indices:", fer_train.class_indices)
print(f"FER-2013 — Train samples: {fer_train.samples}, Test samples: {fer_test.samples}")

Found 6559 images belonging to 6 classes.
Found 7067 images belonging to 6 classes.
FER-2013 class indices: {'angry': 0, 'fear': 1, 'happy': 2, 'neutral': 3, 'sad': 4, 'surprise': 5}
FER-2013 — Train samples: 6559, Test samples: 7067


In [ ]:
def build_custom_cnn(num_classes=NUM_CLASSES):
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', padding='same',
               input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        MaxPooling2D(2,2),
        Dropout(0.25),

        Conv2D(64, (3,3), activation='relu', padding='same'),
        MaxPooling2D(2,2),
        Dropout(0.25),

        Conv2D(128, (3,3), activation='relu', padding='same'),
        MaxPooling2D(2,2),
        Dropout(0.25),

        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    model.summary()
    return model

print("Custom CNN builder defined.")

Custom CNN builder defined.


In [ ]:
def build_vgg16(num_classes=NUM_CLASSES):
    base = VGG16(weights='imagenet',
                 include_top=False,
                 input_shape=(IMG_SIZE, IMG_SIZE, 3))
    # Freeze all base layers
    for layer in base.layers:
        layer.trainable = False

    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    out = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base.input, outputs=out)
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    model.summary()
    return model

print("VGG16 builder defined.")

VGG16 builder defined.


In [ ]:
def save_training_curves(history, model_name, dataset_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Accuracy
    axes[0].plot(history.history['accuracy'],     label='Train',      color='steelblue')
    axes[0].plot(history.history['val_accuracy'], label='Validation', color='orange')
    axes[0].set_title(f'Training Accuracy ({model_name} {dataset_name})')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()

    # Loss
    axes[1].plot(history.history['loss'],     label='Train',      color='steelblue')
    axes[1].plot(history.history['val_loss'], label='Validation', color='orange')
    axes[1].set_title(f'Training Loss ({model_name} {dataset_name})')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    filename = f'{RESULTS}/figure_training_curves_{model_name}_{dataset_name}.png'
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {filename}")

In [ ]:
def save_confusion_matrix(model, test_generator, model_name, dataset_name):
    class_labels = list(test_generator.class_indices.keys())

    # Get predictions
    test_generator.reset()
    preds = model.predict(test_generator, verbose=1)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_generator.classes

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix ({model_name} {dataset_name})')
    plt.tight_layout()
    filename = f'{RESULTS}/figure_confusion_matrix_{model_name}_{dataset_name}.png'
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {filename}")

    # Classification report
    report = classification_report(y_true, y_pred,
                                   target_names=class_labels,
                                   output_dict=True)
    report_path = f'{RESULTS}/classification_report_{model_name}_{dataset_name}.json'
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)

    print(f"\nClassification Report — {model_name} on {dataset_name}:")
    print(classification_report(y_true, y_pred, target_names=class_labels))
    print(f"Saved report: {report_path}")

    return report['accuracy']

In [ ]:
import cv2
import random
from pathlib import Path

def save_sample_predictions(model, test_generator, model_name, dataset_name, n=3):
    class_labels   = list(test_generator.class_indices.keys())
    test_dir       = test_generator.directory

    # Collect n random image paths with their true labels
    selected = []
    for label in random.sample(class_labels, min(n, len(class_labels))):
        label_dir = Path(test_dir) / label
        files     = list(label_dir.glob('*.jpg')) + \
                    list(label_dir.glob('*.png')) + \
                    list(label_dir.glob('*.jpeg'))
        if files:
            selected.append((random.choice(files), label))

    # If not enough variety, fill remaining slots
    while len(selected) < n:
        label     = random.choice(class_labels)
        label_dir = Path(test_dir) / label
        files     = list(label_dir.glob('*.jpg')) + \
                    list(label_dir.glob('*.png')) + \
                    list(label_dir.glob('*.jpeg'))
        if files:
            selected.append((random.choice(files), label))

    fig, axes = plt.subplots(1, n, figsize=(7*n, 7))

    for i, (img_path, true_label) in enumerate(selected):

        # Load ORIGINAL full resolution image from disk
        img_original = cv2.imread(str(img_path))
        img_original = cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB)

        # Upscale to 300x300 with best interpolation for display
        img_display  = cv2.resize(img_original, (300, 300),
                                  interpolation=cv2.INTER_LANCZOS4)

        # Separately prepare 48x48 version for model prediction
        img_model    = cv2.resize(img_original, (IMG_SIZE, IMG_SIZE),
                                  interpolation=cv2.INTER_LANCZOS4)
        img_model    = img_model.astype('float32') / 255.0
        img_model    = np.expand_dims(img_model, axis=0)

        # Predict
        pred         = model.predict(img_model, verbose=0)
        pred_label   = class_labels[np.argmax(pred)]
        confidence   = np.max(pred) * 100

        # Display original quality image
        axes[i].imshow(img_display)
        axes[i].set_title(
            f'Predicted: {pred_label} ({confidence:.1f}%)\nActual: {true_label}',
            fontsize=13, fontweight='bold', pad=12
        )
        axes[i].axis('off')

        # Green border = correct, Red border = wrong
        colour = 'green' if pred_label == true_label else 'red'
        for spine in axes[i].spines.values():
            spine.set_edgecolor(colour)
            spine.set_linewidth(5)
            spine.set_visible(True)

    plt.suptitle(
        f'Sample Predictions — {model_name} on {dataset_name}',
        fontsize=16, fontweight='bold', y=1.02
    )
    plt.tight_layout()

    # Save to Drive at highest quality
    filename = f'{RESULTS}/figure_sample_predictions_{model_name}_{dataset_name}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')

    plt.show()
    plt.close()
    print(f"Saved to Drive: {filename}")

In [ ]:
print("=" * 50)
print("Training Custom CNN on CK+")
print("=" * 50)

cnn_ck = build_custom_cnn()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    ModelCheckpoint(f'{RESULTS}/best_cnn_ck.keras',
                    monitor='val_accuracy', save_best_only=True)
]

history_cnn_ck = cnn_ck.fit(
    ck_train,
    validation_data=ck_test,
    epochs=EPOCHS,
    callbacks=callbacks
)

save_training_curves(history_cnn_ck, 'CNN', 'CK')
acc_cnn_ck = save_confusion_matrix(cnn_ck, ck_test, 'CNN', 'CK')
save_sample_predictions(cnn_ck, ck_test, 'CNN', 'CK')

print(f"\nCNN on CK+ Test Accuracy: {acc_cnn_ck*100:.2f}%")

Training Custom CNN on CK+


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 48, 48, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,274,694 (4.86 MB)

 Trainable params: 1,274,694 (4.86 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 106s 10s/step - accuracy: 0.2141 - loss: 1.7257 - val_accuracy: 0.2657 - val_loss: 1.6932
Epoch 2/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 4s 396ms/step - accuracy: 0.2757 - loss: 1.7018 - val_accuracy: 0.2937 - val_loss: 1.6996
Epoch 3/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 4s 389ms/step - accuracy: 0.2493 - loss: 1.7025 - val_accuracy: 0.2517 - val_loss: 1.7199
Epoch 4/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 7s 601ms/step - accuracy: 0.2463 - loss: 1.6995 - val_accuracy: 0.2657 - val_loss: 1.7183
Epoch 5/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 7s 648ms/step - accuracy: 0.3021 - loss: 1.6753 - val_accuracy: 0.3217 - val_loss: 1.7121
Epoch 6/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 9s 799ms/step - accuracy: 0.2317 - loss: 1.7139 - val_accuracy: 0.2657 - val_loss: 1.7132
Epoch 7/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 5s 412ms/step - accuracy: 0.2903 - loss: 1.6827 - val_accuracy: 0.2867 - val_loss: 1.7153
Epoch 8/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 4s 411ms/step - accuracy: 0.2522 - loss: 1.6790 - val_accuracy: 0.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

Saved to Drive: /content/drive/MyDrive/FER_project/RESULTS/figure_sample_predictions_CNN_CK.png

CNN on CK+ Test Accuracy: 26.57%


In [ ]:
save_sample_predictions(cnn_ck, ck_test, 'CNN', 'CK')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
Saved to Drive: /content/drive/MyDrive/FER_project/RESULTS/figure_sample_predictions_CNN_CK.png


In [ ]:
print("=" * 50)
print("Training Custom CNN on FER-2013 (Fast Version)")
print("=" * 50)

cnn_fer = build_custom_cnn()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint(f'{RESULTS}/best_cnn_fer.keras',
                    monitor='val_accuracy', save_best_only=True)
]

history_cnn_fer = cnn_fer.fit(
    fer_train,
    validation_data=fer_test,
    epochs=15,              # reduced from 30
    steps_per_epoch=200,    # limits batches per epoch instead of full dataset
    validation_steps=50,
    callbacks=callbacks
)

save_training_curves(history_cnn_fer, 'CNN', 'FER2013')
acc_cnn_fer = save_confusion_matrix(cnn_fer, fer_test, 'CNN', 'FER2013')
save_sample_predictions(cnn_fer, fer_test, 'CNN', 'FER2013')

print(f"\nCNN on FER-2013 Test Accuracy: {acc_cnn_fer*100:.2f}%")

Training Custom CNN on FER-2013 (Fast Version)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 48, 48, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,274,694 (4.86 MB)

 Trainable params: 1,274,694 (4.86 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
  4/200 ━━━━━━━━━━━━━━━━━━━━ 24:25 7s/step - accuracy: 0.0690 - loss: 1.8656

KeyboardInterrupt: 

In [ ]:
save_sample_predictions(cnn_fer, fer_test, 'CNN', 'FER2013')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step
Saved to Drive: /content/drive/MyDrive/FER_project/RESULTS/figure_sample_predictions_CNN_FER2013.png


In [ ]:
print("=" * 50)
print("Training VGG16 on CK+")
print("=" * 50)

vgg_ck = build_vgg16()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    ModelCheckpoint(f'{RESULTS}/best_vgg16_ck.keras',
                    monitor='val_accuracy', save_best_only=True)
]

history_vgg_ck = vgg_ck.fit(
    ck_train,
    validation_data=ck_test,
    epochs=EPOCHS,
    callbacks=callbacks
)

save_training_curves(history_vgg_ck, 'VGG16', 'CK')
acc_vgg_ck = save_confusion_matrix(vgg_ck, ck_test, 'VGG16', 'CK')
save_sample_predictions(vgg_ck, ck_test, 'VGG16', 'CK')

print(f"\nVGG16 on CK+ Test Accuracy: {acc_vgg_ck*100:.2f}%")

Training VGG16 on CK+
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 48, 48, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 48, 48, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 48, 48, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 24, 24, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 12, 12, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 12, 12, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 6, 6, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 3, 3, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 3, 3, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 3, 3, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 3, 3, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 1, 1, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,847,558 (56.64 MB)

 Trainable params: 132,870 (519.02 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

Epoch 1/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - accuracy: 0.1525 - loss: 2.1224 - val_accuracy: 0.2028 - val_loss: 1.8491
Epoch 2/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - accuracy: 0.1994 - loss: 1.9596 - val_accuracy: 0.2238 - val_loss: 1.7698
Epoch 3/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 22s 2s/step - accuracy: 0.2229 - loss: 1.8967 - val_accuracy: 0.2378 - val_loss: 1.7226
Epoch 4/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step - accuracy: 0.2258 - loss: 1.8488 - val_accuracy: 0.2797 - val_loss: 1.6974
Epoch 5/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 19s 1s/step - accuracy: 0.2669 - loss: 1.7737 - val_accuracy: 0.2657 - val_loss: 1.6832
Epoch 6/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - accuracy: 0.2786 - loss: 1.7055 - val_accuracy: 0.2867 - val_loss: 1.6734
Epoch 7/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - accuracy: 0.2903 - loss: 1.7622 - val_accuracy: 0.2797 - val_loss: 1.6651
Epoch 8/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 16s 2s/step - accuracy: 0.3079 - loss: 1.6894 - val_accuracy: 0.3077 - val_loss:

5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 701ms/step
Saved: /content/drive/MyDrive/FER_project/RESULTS/figure_confusion_matrix_VGG16_CK.png

Classification Report — VGG16 on CK:
              precision    recall  f1-score   support

       anger       0.33      0.06      0.10        18
        fear       0.00      0.00      0.00         7
       happy       0.55      0.77      0.64        30
     neutral       0.29      0.56      0.38        36
     sadness       0.00      0.00      0.00        17
    surprise       0.48      0.40      0.44        35

    accuracy                           0.41       143
   macro avg       0.28      0.30      0.26       143
weighted avg       0.35      0.41      0.35       143

Saved report: /content/drive/MyDrive/FER_project/RESULTS/classification_report_VGG16_CK.json


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 491ms/step
Saved: /content/drive/MyDrive/FER_project/RESULTS/figure_sample_predictions_VGG16_CK.png

VGG16 on CK+ Test Accuracy: 40.56%


In [ ]:
save_sample_predictions(vgg_ck, ck_test, 'VGG16', 'CK')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 425ms/step
Saved to Drive: /content/drive/MyDrive/FER_project/RESULTS/figure_sample_predictions_VGG16_CK.png


In [ ]:
print("=" * 50)
print("Training VGG16 on FER-2013 (Fast Version)")
print("=" * 50)

vgg_fer = build_vgg16()

# Unfreeze last 4 layers of VGG16 for better accuracy on FER data
base_model = vgg_fer.layers[0] if hasattr(vgg_fer.layers[0], 'layers') else None
for layer in vgg_fer.layers:
    if hasattr(layer, 'layers'):  # this is the VGG16 base
        for vgg_layer in layer.layers[-4:]:
            vgg_layer.trainable = True

# Recompile after unfreezing
vgg_fer.compile(
    optimizer=Adam(learning_rate=1e-5),  # lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint(f'{RESULTS}/best_vgg16_fer.keras',
                    monitor='val_accuracy', save_best_only=True)
]

history_vgg_fer = vgg_fer.fit(
    fer_train,
    validation_data=fer_test,
    epochs=15,               # reduced from 30
    steps_per_epoch=200,     # limits batches per epoch
    validation_steps=50,
    callbacks=callbacks
)

save_training_curves(history_vgg_fer, 'VGG16', 'FER2013')
acc_vgg_fer = save_confusion_matrix(vgg_fer, fer_test, 'VGG16', 'FER2013')
save_sample_predictions(vgg_fer, fer_test, 'VGG16', 'FER2013')

print(f"\nVGG16 on FER-2013 Test Accuracy: {acc_vgg_fer*100:.2f}%")

Training VGG16 on FER-2013 (Fast Version)


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 48, 48, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 48, 48, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 48, 48, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 24, 24, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 12, 12, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 12, 12, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 6, 6, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 3, 3, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 3, 3, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 3, 3, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 3, 3, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 1, 1, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,847,558 (56.64 MB)

 Trainable params: 132,870 (519.02 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

Epoch 1/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 257s 1s/step - accuracy: 0.1697 - loss: 2.0567 - val_accuracy: 0.0894 - val_loss: 1.8801
Epoch 2/15
  5/200 ━━━━━━━━━━━━━━━━━━━━ 4:10 1s/step - accuracy: 0.1384 - loss: 2.0161

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


200/200 ━━━━━━━━━━━━━━━━━━━━ 82s 406ms/step - accuracy: 0.1375 - loss: 2.0405 - val_accuracy: 0.0881 - val_loss: 1.8791
Epoch 3/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 322s 1s/step - accuracy: 0.1724 - loss: 1.9767 - val_accuracy: 0.1044 - val_loss: 1.8412
Epoch 4/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 54s 263ms/step - accuracy: 0.2000 - loss: 1.9429 - val_accuracy: 0.1069 - val_loss: 1.8379
Epoch 5/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 238s 1s/step - accuracy: 0.1742 - loss: 1.9331 - val_accuracy: 0.1100 - val_loss: 1.8337
Epoch 6/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 50s 246ms/step - accuracy: 0.1813 - loss: 1.9209 - val_accuracy: 0.1094 - val_loss: 1.8336
Epoch 7/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 234s 1s/step - accuracy: 0.1771 - loss: 1.9194 - val_accuracy: 0.1000 - val_loss: 1.8328
Epoch 8/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 83s 415ms/step - accuracy: 0.2438 - loss: 1.8118 - val_accuracy: 0.1000 - val_loss: 1.8324
Epoch 9/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 260s 1s/step - accuracy: 0.1938 - loss: 1.9011 - val_accurac

In [ ]:
save_sample_predictions(vgg_fer, fer_test, 'VGG16', 'FER2013')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step
Saved to Drive: /content/drive/MyDrive/FER_project/RESULTS/figure_sample_predictions_VGG16_FER2013.png


In [ ]:
# ── Final model comparison chart ─────────────────────────────────────────────
models    = ['CNN\nCK+', 'CNN\nFER-2013', 'VGG16\nCK+', 'VGG16\nFER-2013']
accuracies = [
    acc_cnn_ck  * 100,
    acc_cnn_fer * 100,
    acc_vgg_ck  * 100,
    acc_vgg_fer * 100
]
colors = ['#2E75B6', '#70AD47', '#2E75B6', '#70AD47']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(models, accuracies, color=colors, edgecolor='white', width=0.5)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.8,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Model Comparison — CNN vs VGG16 on CK+ and FER-2013', fontsize=13)
ax.set_ylim(0, 100)
ax.axhline(y=64, color='red', linestyle='--', linewidth=1.2,
           label='Assignment 1 Baseline (HOG+RF = 64%)')
ax.legend(fontsize=10)
plt.tight_layout()

filename = f'{RESULTS}/figure_model_comparison.png'
plt.savefig(filename, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {filename}")

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "="*55)
print(f"{'FINAL RESULTS SUMMARY':^55}")
print("="*55)
print(f"{'Model':<30} {'Dataset':<12} {'Accuracy':>10}")
print("-"*55)
print(f"{'HOG + RF (Assignment 1)':<30} {'CK+':<12} {'64.00%':>10}")
print(f"{'Custom CNN':<30} {'CK+':<12} {acc_cnn_ck*100:>9.2f}%")
print(f"{'Custom CNN':<30} {'FER-2013':<12} {acc_cnn_fer*100:>9.2f}%")
print(f"{'VGG16 Transfer Learning':<30} {'CK+':<12} {acc_vgg_ck*100:>9.2f}%")
print(f"{'VGG16 Transfer Learning':<30} {'FER-2013':<12} {acc_vgg_fer*100:>9.2f}%")
print("="*55)

# Save summary as text file
summary_path = f'{RESULTS}/results_summary.txt'
with open(summary_path, 'w') as f:
    f.write("FINAL RESULTS SUMMARY\n")
    f.write("="*55 + "\n")
    f.write(f"{'Model':<30} {'Dataset':<12} {'Accuracy':>10}\n")
    f.write("-"*55 + "\n")
    f.write(f"{'HOG + RF (Assignment 1)':<30} {'CK+':<12} {'64.00%':>10}\n")
    f.write(f"{'Custom CNN':<30} {'CK+':<12} {acc_cnn_ck*100:>9.2f}%\n")
    f.write(f"{'Custom CNN':<30} {'FER-2013':<12} {acc_cnn_fer*100:>9.2f}%\n")
    f.write(f"{'VGG16 Transfer Learning':<30} {'CK+':<12} {acc_vgg_ck*100:>9.2f}%\n")
    f.write(f"{'VGG16 Transfer Learning':<30} {'FER-2013':<12} {acc_vgg_fer*100:>9.2f}%\n")

print(f"\nSummary saved to: {summary_path}")
print(f"\nAll results saved in: {RESULTS}")

Saved: /content/drive/MyDrive/FER_project/RESULTS/figure_model_comparison.png

                 FINAL RESULTS SUMMARY                 
Model                          Dataset        Accuracy
-------------------------------------------------------
HOG + RF (Assignment 1)        CK+              64.00%
Custom CNN                     CK+              25.17%
Custom CNN                     FER-2013         23.52%
VGG16 Transfer Learning        CK+              40.56%
VGG16 Transfer Learning        FER-2013         24.35%

Summary saved to: /content/drive/MyDrive/FER_project/RESULTS/results_summary.txt

All results saved in: /content/drive/MyDrive/FER_project/RESULTS


In [ ]:
save_sample_predictions(cnn_ck,  ck_test,  'CNN',   'CK')
save_sample_predictions(cnn_fer, fer_test, 'CNN',   'FER2013')
save_sample_predictions(vgg_ck,  ck_test,  'VGG16', 'CK')
save_sample_predictions(vgg_fer, fer_test, 'VGG16', 'FER2013')

Saved to Drive: /content/drive/MyDrive/FER_project/RESULTS/figure_sample_predictions_CNN_CK.png
Saved to Drive: /content/drive/MyDrive/FER_project/RESULTS/figure_sample_predictions_CNN_FER2013.png
Saved to Drive: /content/drive/MyDrive/FER_project/RESULTS/figure_sample_predictions_VGG16_CK.png
Saved to Drive: /content/drive/MyDrive/FER_project/RESULTS/figure_sample_predictions_VGG16_FER2013.png


In [ ]:
import matplotlib.pyplot as plt
import os
import random
import cv2

emotions = ['anger', 'fear', 'happy', 'neutral', 'sadness', 'surprise']
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for i, emotion in enumerate(emotions):
    folder = os.path.join(CK_TRAIN, emotion)
    files = os.listdir(folder)
    img_path = os.path.join(folder, random.choice(files))
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (200, 200))
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(emotion.capitalize(), fontsize=14, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Sample Images from CK+ Dataset — Six Emotion Classes',
             fontsize=16, fontweight='bold')
plt.tight_layout()
filename = f'{RESULTS}/figure_dataset_samples_CK.png'
plt.savefig(filename, dpi=200, bbox_inches='tight')
plt.show()
plt.close()
print(f'Saved: {filename}')

Saved: /content/drive/MyDrive/FER_project/RESULTS/figure_dataset_samples_CK.png


In [ ]:
import os
print(os.listdir(FER_TRAIN))

['angry', 'neutral', 'happy', 'fear', 'surprise', 'sad']


In [ ]:
import matplotlib.pyplot as plt
import os
import random
import cv2

emotions = ['anger', 'fear', 'happy', 'neutral', 'sadness', 'surprise']

# ── CK+ row ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 6, figsize=(18, 4))
for i, emotion in enumerate(emotions):
    folder = os.path.join(CK_TRAIN, emotion)
    files = [f for f in os.listdir(folder) if f.endswith(('.jpg','.png','.jpeg'))]
    img_path = os.path.join(folder, random.choice(files))
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (200, 200), interpolation=cv2.INTER_LANCZOS4)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(emotion.capitalize(), fontsize=13, fontweight='bold', pad=8)
    axes[i].axis('off')

plt.suptitle('Sample Images from CK+ Dataset — Six Emotion Classes',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
filename = f'{RESULTS}/figure_dataset_samples_CK.png'
plt.savefig(filename, dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
plt.close()
print(f'Saved: {filename}')

# ── FER-2013 row ──────────────────────────────────────────────────────────────
fer_emotions = ['angry', 'fear', 'happy', 'neutral', 'sad', 'surprise']

fig, axes = plt.subplots(1, 6, figsize=(18, 4))
for i, emotion in enumerate(fer_emotions):
    folder = os.path.join(FER_TRAIN, emotion)
    files = [f for f in os.listdir(folder) if f.endswith(('.jpg','.png','.jpeg'))]
    img_path = os.path.join(folder, random.choice(files))
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (200, 200), interpolation=cv2.INTER_LANCZOS4)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(emotion.capitalize(), fontsize=13, fontweight='bold', pad=8)
    axes[i].axis('off')

plt.suptitle('Sample Images from FER-2013 Dataset — Six Emotion Classes',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
filename = f'{RESULTS}/figure_dataset_samples_FER2013.png'
plt.savefig(filename, dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
plt.close()
print(f'Saved: {filename}')

Saved: /content/drive/MyDrive/FER_project/RESULTS/figure_dataset_samples_CK.png
Saved: /content/drive/MyDrive/FER_project/RESULTS/figure_dataset_samples_FER2013.png


In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.models import load_model

RESULTS = '/content/drive/MyDrive/FER_project/RESULTS'

In [ ]:
cnn_ck  = load_model(f'{RESULTS}/best_cnn_ck.keras')
vgg_ck  = load_model(f'{RESULTS}/best_vgg16_ck.keras')
print("Models loaded")

Models loaded


In [ ]:
# Print exact layer architecture
print("CNN Architecture:")
cnn_ck.summary()

print("\nVGG16 Model Architecture:")
vgg_ck.summary()

CNN Architecture:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 48, 48, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,824,084 (14.59 MB)

 Trainable params: 1,274,694 (4.86 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 48, 48, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 48, 48, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 48, 48, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 24, 24, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 12, 12, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 12, 12, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 6, 6, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 3, 3, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 3, 3, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 3, 3, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 3, 3, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 1, 1, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,113,300 (57.65 MB)

 Trainable params: 132,870 (519.02 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

 Optimizer params: 265,742 (1.01 MB)